# 0.1. Delete baseline

This baseline is based on the idea of simply deleting bad words from the text.


In [8]:
import pandas as pd

In [9]:
# Load the bad bad words dataset
bad_bad_words = pd.read_csv("../data/external/bad-words.csv", names=["word"])
bad_bad_words.head()

,word
0,jigaboo
1,mound of venus
2,asslover
3,s&m
4,queaf


In [10]:
def kgram(word, k=3):
    """Return a list of k-grams for a given word."""
    return [word[i : i + k] for i in range(len(word) - k + 1)]


def kgram_similarity(word1, word2, k=3):
    """Return the Jaccard similarity between two words."""
    A = set(kgram(word1, k))
    B = set(kgram(word2, k))

    inter_len = len(A & B)
    union_len = len(A | B)

    if union_len == 0:
        return 0

    return inter_len / union_len


def is_bad_word(word):
    """Use k-gram similarity to determine if a word is bad."""
    return any(
        kgram_similarity(word, bad_word, 3) > 0.5 for bad_word in bad_bad_words["word"]
    )


def clean_word(word):
    """Remove symbols from a word."""
    cleaned_word = "".join(c for c in word if c.isalpha())
    cleaned_word = cleaned_word.strip()
    return cleaned_word


def detoxify(text):
    """Replace bad words in a text with asterisks."""
    words = text.split()

    for i, word in enumerate(words):
        cleaned_word = clean_word(word).lower()

        if is_bad_word(cleaned_word):
            words[i] = "*" * len(word)

    return " ".join(words)

In [11]:
# Get random toxic sentenc from the dataset
df_test = pd.read_csv(
    "../data/interim/processed.tsv", sep="\t", header=None, names=["tox", "detox"]
)
df_test.head()

,tox,detox
0,"if Alkar floods her with her mental waste, it ...","If Alkar is flooding her with psychic waste, t..."
1,you're becoming disgusting.,Now you're getting nasty.
2,"well, we can spare your life.","Well, we could spare your life, for one."
3,"monkey, you have to wake up.","Ah! Monkey, you've got to snap out of it."
4,I have orders to kill her.,I've got orders to put her down.


In [12]:
import random

In [15]:
tox_infer = df_test["tox"][random.randint(0, len(df_test))]

print(f"Tox:\t{tox_infer}\n\nDelete:\t{detoxify(tox_infer)}")

Tox:	Master, I can't take women.

Delete:	Master, I can't take ******
